# Lab 4: Representation Learning

Adapted from 6.7960 Fall 2025 HW4.

A model can reconstruct pixels accurately without arranging its
embedding around distinctions useful for a downstream task. You will
train two complete representation learners, evaluate their frozen
embeddings with k-nearest neighbors and linear probes, and test how
augmentation choices determine what each embedding preserves.

The source notebook downloads a roughly 787 MB shapes asset and
schedules several long runs. This online adaptation creates a compact
colored-shapes dataset locally, supplies every implementation, and
supports optional checkpoints. No factor labels are used to train the
encoders; labels are exposed only to held-out probes. Measured probe
scores are not graded values.


## Runtime and deterministic setup

QUICK mode uses 4,096 training examples and three epochs. Set
DL_LAB_MODE to full before running this cell for 16,384 examples and
twelve epochs.


In [ ]:
import os, random, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 7960
MODE = os.environ.get("DL_LAB_MODE", "quick").lower()
if MODE not in {"quick", "full"}:
    raise ValueError("DL_LAB_MODE must be quick or full")
QUICK_MODE = MODE == "quick"

def seed_all(seed=SEED):
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except AttributeError:
        pass

seed_all()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SETTINGS = {
    "quick": dict(train_n=4096, val_n=1024, epochs=3, probe_epochs=80),
    "full": dict(train_n=16384, val_n=4096, epochs=12, probe_epochs=150),
}[MODE]
BATCH_SIZE, LATENT_DIM = 256, 128
print("mode:", MODE, "device:", DEVICE, "settings:", SETTINGS)


## Compact colored-shapes data

Each 32 by 32 image contains a circle, square, or triangle in red,
green, or blue. Shape, color, position, size, and background noise vary
independently. The split and index completely determine an image.


In [ ]:
SHAPES = ("circle", "square", "triangle")
COLOR_NAMES = ("red", "green", "blue")
COLORS = ((225, 60, 60), (55, 190, 95), (65, 105, 225))

def render_shape(index, split):
    offset = 0 if split == "train" else 1_000_000
    rng = np.random.default_rng(SEED + offset + index)
    shape = index % 3
    color = (index // 3) % 3
    background = rng.integers(8, 28, (32, 32, 3), dtype=np.uint8)
    image = Image.fromarray(background, "RGB")
    draw = ImageDraw.Draw(image)
    size = int(rng.integers(12, 20)); half = size // 2
    cx = int(rng.integers(half + 2, 32 - half - 1))
    cy = int(rng.integers(half + 2, 32 - half - 1))
    box = (cx - half, cy - half, cx + half, cy + half)
    if shape == 0:
        draw.ellipse(box, fill=COLORS[color])
    elif shape == 1:
        draw.rectangle(box, fill=COLORS[color])
    else:
        draw.polygon(
            [(cx, cy - half), (cx - half, cy + half), (cx + half, cy + half)],
            fill=COLORS[color],
        )
    return np.asarray(image).copy(), shape, color

class ColoredShapes(Dataset):
    def __init__(self, count, split):
        self.count, self.split = count, split
    def __len__(self):
        return self.count
    def __getitem__(self, index):
        image, shape, color = render_shape(index, self.split)
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255
        return image, shape, color

train_data = ColoredShapes(SETTINGS["train_n"], "train")
val_data = ColoredShapes(SETTINGS["val_n"], "validation")

def eval_loader(data, shuffle=False):
    return DataLoader(
        data, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
        pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(SEED),
    )

train_eval_loader = eval_loader(train_data)
val_eval_loader = eval_loader(val_data)
images, shape_labels, color_labels = next(iter(val_eval_loader))
fig, axes = plt.subplots(2, 6, figsize=(9, 3.3))
for ax, image, shape, color in zip(
    axes.flat, images[:12], shape_labels[:12], color_labels[:12]
):
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(SHAPES[shape] + " / " + COLOR_NAMES[color], fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()


## Prefilled autoencoder and contrastive encoder

Both objectives use the same compact convolutional encoder and a
128-dimensional latent. The autoencoder adds a decoder and minimizes
pixel MSE. The contrastive encoder L2-normalizes its output.


In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent=LATENT_DIM, normalize=False):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.projection = nn.Linear(128 * 4 * 4, latent)
        self.normalize = normalize
    def forward(self, x):
        z = self.projection(self.features(x).flatten(1))
        return F.normalize(z, dim=1) if self.normalize else z

class Decoder(nn.Module):
    def __init__(self, latent=LATENT_DIM):
        super().__init__()
        self.input = nn.Linear(latent, 128 * 4 * 4)
        self.layers = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )
    def forward(self, z):
        return self.layers(self.input(z).view(-1, 128, 4, 4))

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder, self.decoder = Encoder(), Decoder()
    def forward(self, x):
        return self.decoder(self.encoder(x))

def nparams(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

seed_all()
autoencoder = Autoencoder().to(DEVICE)
seed_all()
contrastive_encoder = Encoder(normalize=True).to(DEVICE)
print("autoencoder parameters:", nparams(autoencoder))
print("contrastive encoder parameters:", nparams(contrastive_encoder))


## Augmentations define invariances

A positive pair contains two transformed views of one image. Strong
recoloring rewards color invariance. Geometry changes reward invariance
to position, crop, and orientation. The balanced policy is the
baseline.


In [ ]:
def augmentation(name):
    choices = {
        "balanced": transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(.75, 1), ratio=(.9, 1.1)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomApply([
                transforms.ColorJitter(.15, .15, .15, .03)
            ], p=.5),
        ]),
        "color": transforms.Compose([
            transforms.RandomResizedCrop(32, scale=(.8, 1), ratio=(.95, 1.05)),
            transforms.RandomGrayscale(p=.35),
            transforms.RandomApply([
                transforms.ColorJitter(.35, .35, .8, .25)
            ], p=.9),
        ]),
        "geometry": transforms.Compose([
            transforms.Pad(4, padding_mode="reflect"),
            transforms.RandomRotation(35),
            transforms.RandomResizedCrop(32, scale=(.65, 1), ratio=(.85, 1.15)),
        ]),
    }
    if name not in choices:
        raise KeyError(name)
    return choices[name]

class PositivePairs(Dataset):
    def __init__(self, base, transform):
        self.base, self.transform = base, transform
    def __len__(self):
        return len(self.base)
    def __getitem__(self, index):
        image, _, _ = self.base[index]
        return self.transform(image), self.transform(image)

BASE_AUGMENTATION = "balanced"
pair_data = PositivePairs(train_data, augmentation(BASE_AUGMENTATION))
pair_loader = DataLoader(
    pair_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=0, pin_memory=DEVICE.type == "cuda",
    generator=torch.Generator().manual_seed(SEED),
)

diagnostic_left, diagnostic_right = next(iter(pair_loader))
NEGATIVES_PER_ANCHOR = diagnostic_left.shape[0] - 1
assert diagnostic_left.shape[0] == 256
assert NEGATIVES_PER_ANCHOR == 255
print("FULL CONTRASTIVE BATCH =", diagnostic_left.shape[0])
print("DETERMINISTIC NEGATIVES PER ANCHOR =", NEGATIVES_PER_ANCHOR)


## Losses, short training loops, and optional checkpoints

Following the source notebook, each anchor selects its match from a
256 by 256 cross-view similarity matrix. The matching diagonal is one
positive and the other 255 entries are negatives. Checkpoint paths may
be blank; no reference output is fabricated when a checkpoint is
absent.


In [ ]:
def contrastive_loss(encoder, left, right, temperature=.07):
    z1, z2 = encoder(left), encoder(right)
    logits = z1 @ z2.T / temperature
    target = torch.arange(len(left), device=left.device)
    return .5 * (
        F.cross_entropy(logits, target) +
        F.cross_entropy(logits.T, target)
    )

def load_if_present(path, model):
    if not path:
        return False
    path = Path(path)
    if not path.exists():
        print("checkpoint not found, training from scratch:", path)
        return False
    payload = torch.load(path, map_location=DEVICE)
    model.load_state_dict(payload["model_state"])
    print("loaded:", path, payload.get("metadata", {}))
    return True

def save_checkpoint(path, model, metadata=None):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model_state": model.state_dict(), "metadata": metadata or {}}, path)
    print("saved:", path)

def train_autoencoder(model, epochs):
    seed_all()
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    history = []; source = eval_loader(train_data, True)
    for epoch in range(1, epochs + 1):
        model.train(); total = count = 0; started = time.perf_counter()
        for images, _, _ in source:
            images = images.to(DEVICE); opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(images), images)
            loss.backward(); opt.step()
            total += loss.item() * len(images); count += len(images)
        row = dict(epoch=epoch, mse=total / count, seconds=time.perf_counter()-started)
        history.append(row); print("autoencoder", row)
    return history

def train_contrastive(model, source, epochs):
    seed_all()
    opt = torch.optim.Adam(model.parameters(), lr=2e-4)
    history = []
    for epoch in range(1, epochs + 1):
        model.train(); total = batches = 0; started = time.perf_counter()
        for left, right in source:
            left, right = left.to(DEVICE), right.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = contrastive_loss(model, left, right)
            loss.backward(); opt.step()
            total += loss.item(); batches += 1
        row = dict(
            epoch=epoch, loss=total / batches,
            seconds=time.perf_counter() - started,
        )
        history.append(row); print("contrastive", row)
    return history

AE_CHECKPOINT = os.environ.get("M4_AE_CHECKPOINT", "")
CONTRASTIVE_CHECKPOINT = os.environ.get("M4_CONTRASTIVE_CHECKPOINT", "")
ae_history = [] if load_if_present(AE_CHECKPOINT, autoencoder) else train_autoencoder(
    autoencoder, SETTINGS["epochs"]
)
contrastive_history = (
    [] if load_if_present(CONTRASTIVE_CHECKPOINT, contrastive_encoder)
    else train_contrastive(contrastive_encoder, pair_loader, SETTINGS["epochs"])
)


## Inspect reconstruction

Reconstruction measures the autoencoder's own objective. It does not
establish that the latent is organized for shape or color decisions.


In [ ]:
@torch.no_grad()
def show_reconstructions(model, count=8):
    model.eval(); images, _, _ = next(iter(val_eval_loader))
    original = images[:count]
    rebuilt = model(original.to(DEVICE)).cpu()
    fig, axes = plt.subplots(2, count, figsize=(1.4 * count, 3))
    for i in range(count):
        axes[0, i].imshow(original[i].permute(1, 2, 0))
        axes[1, i].imshow(rebuilt[i].permute(1, 2, 0).clamp(0, 1))
        axes[0, i].axis("off"); axes[1, i].axis("off")
    axes[0, 0].set_ylabel("input"); axes[1, 0].set_ylabel("rebuild")
    plt.tight_layout(); plt.show()

show_reconstructions(autoencoder)


## Frozen kNN and linear probes

kNN asks whether nearby embeddings share a factor. A linear probe asks
whether one linear decision layer can recover it. Encoders remain
frozen. Report both shape and color so an augmentation's tradeoff is
visible.


In [ ]:
@torch.no_grad()
def features(encoder, source):
    encoder.eval(); xs, shapes, colors = [], [], []
    for images, shape, color in source:
        xs.append(encoder(images.to(DEVICE)).cpu())
        shapes.append(shape); colors.append(color)
    return torch.cat(xs), torch.cat(shapes), torch.cat(colors)

@torch.no_grad()
def knn(train_x, train_y, val_x, val_y, k=5, chunk=256):
    train_x = F.normalize(train_x.float(), dim=1)
    val_x = F.normalize(val_x.float(), dim=1)
    predicted = []
    for start in range(0, len(val_x), chunk):
        near = (val_x[start:start+chunk] @ train_x.T).topk(k, 1).indices
        predicted.append(train_y[near].mode(1).values)
    return (torch.cat(predicted) == val_y).float().mean().item()

def linear_probe(train_x, train_y, val_x, val_y, classes):
    seed_all()
    mean, std = train_x.mean(0, True), train_x.std(0, True).clamp_min(1e-6)
    train_x = ((train_x - mean) / std).to(DEVICE)
    val_x = ((val_x - mean) / std).to(DEVICE)
    train_y, val_y = train_y.to(DEVICE), val_y.to(DEVICE)
    probe = nn.Linear(train_x.shape[1], classes).to(DEVICE)
    opt = torch.optim.SGD(probe.parameters(), lr=.2, momentum=.9, weight_decay=1e-4)
    for _ in range(SETTINGS["probe_epochs"]):
        opt.zero_grad(set_to_none=True)
        loss = F.cross_entropy(probe(train_x), train_y)
        loss.backward(); opt.step()
    with torch.no_grad():
        return (probe(val_x).argmax(1) == val_y).float().mean().item()

FEATURE_SETS = {}
for name, encoder in {
    "autoencoder": autoencoder.encoder,
    "contrastive": contrastive_encoder,
}.items():
    train_set = features(encoder, train_eval_loader)
    val_set = features(encoder, val_eval_loader)
    FEATURE_SETS[name] = dict(train=train_set, val=val_set)
    tx, ts, tc = train_set; vx, vs, vc = val_set
    print({
        "encoder": name,
        "shape_knn": knn(tx, ts, vx, vs),
        "color_knn": knn(tx, tc, vx, vc),
        "shape_linear_probe": linear_probe(tx, ts, vx, vs, 3),
        "color_linear_probe": linear_probe(tx, tc, vx, vc, 3),
    })


## Inspect neighborhoods

Quantitative probes summarize a split. Neighbor panels reveal the
embedding's notion of similarity example by example.


In [ ]:
@torch.no_grad()
def show_neighbors(name, query=0, count=8):
    train_x = F.normalize(FEATURE_SETS[name]["train"][0].float(), dim=1)
    val_x = F.normalize(FEATURE_SETS[name]["val"][0][query:query+1].float(), dim=1)
    ids = (val_x @ train_x.T).squeeze().topk(count).indices.tolist()
    query_image, query_shape, query_color = val_data[query]
    fig, axes = plt.subplots(1, count + 1, figsize=(1.4 * (count + 1), 2))
    axes[0].imshow(query_image.permute(1, 2, 0))
    axes[0].set_title("query: " + SHAPES[query_shape] + "/" + COLOR_NAMES[query_color], fontsize=8)
    axes[0].axis("off")
    for ax, index in zip(axes[1:], ids):
        image, shape, color = train_data[index]
        ax.imshow(image.permute(1, 2, 0))
        ax.set_title(SHAPES[shape] + " / " + COLOR_NAMES[color], fontsize=8)
        ax.axis("off")
    fig.suptitle(name); plt.tight_layout(); plt.show()

show_neighbors("autoencoder")
show_neighbors("contrastive")


## Controlled augmentation tuning

Choose color or geometry, predict what information it will suppress,
then train a fresh encoder with every other setting held fixed. Set
RUN_TUNING to True when ready.


In [ ]:
CHOSEN_AUGMENTATION, RUN_TUNING = "color", False
tuned_encoder = None
if RUN_TUNING:
    tuned_data = PositivePairs(train_data, augmentation(CHOSEN_AUGMENTATION))
    tuned_loader = DataLoader(
        tuned_data, batch_size=256, shuffle=True, drop_last=True,
        num_workers=0, pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(SEED),
    )
    seed_all(); tuned_encoder = Encoder(normalize=True).to(DEVICE)
    tuned_history = train_contrastive(tuned_encoder, tuned_loader, SETTINGS["epochs"])
    tx, ts, tc = features(tuned_encoder, train_eval_loader)
    vx, vs, vc = features(tuned_encoder, val_eval_loader)
    print({
        "augmentation": CHOSEN_AUGMENTATION,
        "shape_knn": knn(tx, ts, vx, vs),
        "color_knn": knn(tx, tc, vx, vc),
    })
else:
    print("Choose an augmentation and set RUN_TUNING=True.")


## What to record

Record whether reconstruction predicted useful neighborhoods, and
which information your chosen positive-pair transformation encouraged
the model to ignore.

The edX submission grades the deterministic 255 negatives per anchor
and objective/augmentation conclusions that remain valid across noisy
runs. It does not grade a freely measured probe score.

The HW3 ResNet-50 feature-synthesis panel is a valuable but expensive
optional extension. Use instructor-provided cached panels rather than
generating the full multi-hundred-step sweep for this lab.


## Report values

Run the cell below **after** training all three objectives fresh (the
autoencoder and contrastive baselines, then the tuned encoder with
`RUN_TUNING=True`). It prints three fixed, mode-independent integers: a full
contrastive batch holds 256 anchors, each anchor sees 255 in-batch negatives,
and all three training histories reached the configured epoch count. Enter R1,
R2, and R3 in the first Lab 4 submission problem.


In [ ]:
full_contrastive_batch = NEGATIVES_PER_ANCHOR + 1
objective_histories = {
    "autoencoder": ae_history,
    "contrastive": contrastive_history,
    "tuned": tuned_history if RUN_TUNING else [],
}
completed_objectives = sum(
    len(history) == SETTINGS["epochs"] for history in objective_histories.values()
)
report_values = {
    "R1 - anchors in one full contrastive batch": full_contrastive_batch,
    "R2 - in-batch negatives available to each anchor": NEGATIVES_PER_ANCHOR,
    "R3 - complete training histories (autoencoder, contrastive, tuned)": completed_objectives,
}
assert tuple(report_values.values()) == (256, 255, 3)
print("LAB 4 REPORT VALUES (enter integers without separators)")
for label, value in report_values.items():
    print(label + ":", value)
